In [ ]:
import tkinter as tk
import itertools

unit_data = [
    "1,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "2,y = (7E-07*x**2) - (0.0005*x) + 0.3874,0,0",
    "3,y = (-5E-08*x**2) - (0.0001*x) + 0.315,150,350",
    "4,y = (7E-07*x**2) - (0.0005*x) + 0.3869,0,0",
    "5,y = (-5E-07*x**2) + (0.0003*x)+ 0.2885,90,300",
    "6,y = (-4E-06*x**2) + (0.0015*x) + 0.2112,150,330",
    "7,y = (1E-06*x**2) - (0.0006*x) + 0.3904,90,300",
    "8,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "9,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "10,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
]

def fuel_rate_function(fuel_rate_str):
    def fuel_rate(x):
        return eval(fuel_rate_str)
    return fuel_rate

def calc_fuel_consumption(unit, demand):
    fuel_rate = operational_units[unit]["fuel_rate"](demand)
    fuel_consumption = fuel_rate * demand
    return fuel_rate, fuel_consumption

def find_optimal_dispatch(total_demand):
    optimal_dispatch = None
    min_fuel_cost = float("inf")
    closest_demand_diff = float("inf")

    for i in range(1, len(operational_units) + 1):
        for combination in itertools.combinations(operational_units.keys(), i):
            min_capacity = sum(operational_units[unit]["min_capacity"] for unit in combination)
            max_capacity = sum(operational_units[unit]["max_capacity"] for unit in combination)

            if min_capacity <= total_demand and total_demand <= max_capacity:
                fuel_cost = 0
                demand_allocation = []
                remaining_demand = total_demand
                allocated_total_demand = 0

                for unit in combination:
                    if operational_units[unit]["max_capacity"] == 0:
                        continue
                    allocated_demand = max(
                        operational_units[unit]["min_capacity"],
                        min(operational_units[unit]["max_capacity"], remaining_demand)
                    )
                    remaining_demand -= allocated_demand
                    allocated_total_demand += allocated_demand
                    demand_allocation.append((unit, allocated_demand))

                    _, fuel_consumption = calc_fuel_consumption(unit, allocated_demand)
                    fuel_cost += fuel_consumption

                demand_diff = abs(allocated_total_demand - total_demand)
                if demand_diff < closest_demand_diff or (demand_diff == closest_demand_diff and fuel_cost < min_fuel_cost):
                    closest_demand_diff = demand_diff
                    min_fuel_cost = fuel_cost
                    optimal_dispatch = demand_allocation

    return optimal_dispatch, min_fuel_cost

units = {}

root = tk.Tk()
root.title("FRIP - Fuel Rate Improvement Program: Economic Load Dispatch Program")

# Adjust the window size
root.geometry("1400x1050")

# Add titles
title_font = ("Arial", 18, "bold")
subtitle_font = ("Arial", 14)

title = tk.Label(root, text="Fuel Rate Improvement Program (FRIP)", font=title_font, fg='dark green')
title.pack(pady=10)

subtitle = tk.Label(root, text="Economic Load Dispatch Program", font=subtitle_font, fg='red')
subtitle.pack(pady=5)

frame = tk.Frame(root)
frame.pack()

entries = []
unit_count = len(unit_data)
for i in range(1, unit_count + 1):
    unit_number, _, _, _ = unit_data[i - 1].split(',')
    tk.Label(frame, text=f"Is unit {unit_number} operational? (1 for yes, 0 for no): ").grid(row=i - 1, column=0)
    entry = tk.Entry(frame)
    entry.grid(row=i - 1, column=1)
    entries.append(entry)

tk.Label(frame, text="Enter the total demand: ").grid(row=unit_count, column=0)
demand_entry = tk.Entry(frame)
demand_entry.grid(row=unit_count, column=1)

def run_simulation():
    global units
    global operational_units
    for i, data in enumerate(unit_data, start=1):
        unit_number, fuel_rate_str, min_capacity, max_capacity = data.split(',')
        unit_number = int(unit_number)
        min_capacity = float(min_capacity)
        max_capacity = float(max_capacity)
        operational = int(entries[i - 1].get())
        units[unit_number] = {
            "fuel_rate": fuel_rate_function(fuel_rate_str[4:]),
            "min_capacity": min_capacity,
            "max_capacity": max_capacity,
            "operational": operational,
        }
    operational_units = {k: v for k, v in units.items() if v["operational"]}
    total_demand = float(demand_entry.get())
    optimal_dispatch, min_fuel_cost = find_optimal_dispatch(total_demand)

    output_text.insert(tk.END, "Most economical unit(s) to operate:\n")
    for unit, allocated_demand in optimal_dispatch:
        fuel_rate, fuel_consumption = calc_fuel_consumption(unit, allocated_demand)
        fuel_rate = format(fuel_rate, ".4f")
        fuel_consumption = format(fuel_consumption, ".4f")
        output_text.insert(tk.END, f"Unit: {unit}, Allocated Demand: {allocated_demand}, Fuel Rate: {fuel_rate}, Fuel Consumption: {fuel_consumption}\n")
    min_fuel_cost = format(min_fuel_cost, ".4f")
    output_text.insert(tk.END, f"Total Fuel Consumption: {min_fuel_cost}\n")

tk.Button(frame, text="Run Simulation", command=run_simulation).grid(row=unit_count + 1, column=0, columnspan=2)

output_text = tk.Text(root, width=100, height=20)
output_text.pack(pady=10)

root.mainloop()


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\LOkimoki\AppData\Local\Programs\Python\Python313\Lib\tkinter\__init__.py", line 2068, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\LOkimoki\AppData\Local\Temp\ipykernel_60360\3893260717.py", line 119, in run_simulation
    for unit, allocated_demand in optimal_dispatch:
                                  ^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\LOkimoki\AppData\Local\Programs\Python\Python313\Lib\tkinter\__init__.py", line 2068, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\LOkimoki\AppData\Local\Temp\ipykernel_60360\3893260717.py", line 119, in run_simulation
    for unit, allocated_demand in optimal_dispatch:
                                  ^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable
Exception in Tkinter c

In [ ]:
import tkinter as tk
import itertools

unit_data = [
    "1,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "2,y = (7E-07*x**2) - (0.0005*x) + 0.3874,0,0",
    "3,y = (-5E-08*x**2) - (0.0001*x) + 0.315,150,350",
    "4,y = (7E-07*x**2) - (0.0005*x) + 0.3869,0,0",
    "5,y = (-5E-07*x**2) + (0.0003*x)+ 0.2885,90,300",
    "6,y = (-4E-06*x**2) + (0.0015*x) + 0.2112,150,330",
    "7,y = (1E-06*x**2) - (0.0006*x) + 0.3904,90,300",
    "8,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "9,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "10,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
]

def fuel_rate_function(fuel_rate_str):
    def fuel_rate(x):
        return eval(fuel_rate_str)
    return fuel_rate

def calc_fuel_consumption(unit, demand):
    fuel_rate = operational_units[unit]["fuel_rate"](demand)
    fuel_consumption = fuel_rate * demand
    return fuel_rate, fuel_consumption

def find_optimal_dispatch(total_demand):
    optimal_dispatch = None
    min_fuel_cost = float("inf")
    closest_demand_diff = float("inf")

    for i in range(1, len(operational_units) + 1):
        for combination in itertools.combinations(operational_units.keys(), i):
            min_capacity = sum(operational_units[unit]["min_capacity"] for unit in combination)
            max_capacity = sum(operational_units[unit]["max_capacity"] for unit in combination)

            if min_capacity <= total_demand <= max_capacity:
                fuel_cost = 0
                demand_allocation = []
                remaining_demand = total_demand
                allocated_total_demand = 0

                for unit in combination:
                    if operational_units[unit]["max_capacity"] == 0:
                        continue
                    allocated_demand = max(
                        operational_units[unit]["min_capacity"],
                        min(operational_units[unit]["max_capacity"], remaining_demand)
                    )
                    remaining_demand -= allocated_demand
                    allocated_total_demand += allocated_demand
                    demand_allocation.append((unit, allocated_demand))

                    _, fuel_consumption = calc_fuel_consumption(unit, allocated_demand)
                    fuel_cost += fuel_consumption

                demand_diff = abs(allocated_total_demand - total_demand)
                if demand_diff < closest_demand_diff or (demand_diff == closest_demand_diff and fuel_cost < min_fuel_cost):
                    closest_demand_diff = demand_diff
                    min_fuel_cost = fuel_cost
                    optimal_dispatch = demand_allocation

    return optimal_dispatch, min_fuel_cost

units = {}
operational_units = {}

root = tk.Tk()
root.title("FRIP - Fuel Rate Improvement Program: Economic Load Dispatch Program")

# Adjust the window size
root.geometry("1400x1050")

# Add titles
title_font = ("Arial", 18, "bold")
subtitle_font = ("Arial", 14)

title = tk.Label(root, text="Fuel Rate Improvement Program (FRIP)", font=title_font, fg='dark green')
title.pack(pady=10)

subtitle = tk.Label(root, text="Economic Load Dispatch Program", font=subtitle_font, fg='red')
subtitle.pack(pady=5)

frame = tk.Frame(root)
frame.pack()

entries = []
unit_count = len(unit_data)
for i in range(1, unit_count + 1):
    unit_number, _, _, _ = unit_data[i - 1].split(',')
    tk.Label(frame, text=f"Is unit {unit_number} operational? (1 for yes, 0 for no): ").grid(row=i - 1, column=0)
    entry = tk.Entry(frame)
    entry.grid(row=i - 1, column=1)
    entries.append(entry)

tk.Label(frame, text="Enter the total demand: ").grid(row=unit_count, column=0)
demand_entry = tk.Entry(frame)
demand_entry.grid(row=unit_count, column=1)

def run_simulation():
    global units
    global operational_units
    units.clear()
    operational_units.clear()

    for i, data in enumerate(unit_data, start=1):
        try:
            unit_number, fuel_rate_str, min_capacity, max_capacity = data.split(',')
            unit_number = int(unit_number)
            min_capacity = float(min_capacity)
            max_capacity = float(max_capacity)
            operational = int(entries[i - 1].get())
            units[unit_number] = {
                "fuel_rate": fuel_rate_function(fuel_rate_str[4:]),
                "min_capacity": min_capacity,
                "max_capacity": max_capacity,
                "operational": operational,
            }
        except ValueError:
            output_text.insert(tk.END, f"Invalid input for unit {unit_number}. Please enter 1 or 0 for operational status.\n")
            return

    operational_units = {k: v for k, v in units.items() if v["operational"]}
    try:
        total_demand = float(demand_entry.get())
    except ValueError:
        output_text.insert(tk.END, "Invalid total demand. Please enter a numerical value.\n")
        return

    if not operational_units:
        output_text.insert(tk.END, "No operational units available.\n")
        return

    optimal_dispatch, min_fuel_cost = find_optimal_dispatch(total_demand)

    output_text.insert(tk.END, "Most economical unit(s) to operate:\n")
    if optimal_dispatch is None:
        output_text.insert(tk.END, "No feasible dispatch found for the given total demand.\n")
    else:
        for unit, allocated_demand in optimal_dispatch:
            fuel_rate, fuel_consumption = calc_fuel_consumption(unit, allocated_demand)
            fuel_rate = format(fuel_rate, ".4f")
            fuel_consumption = format(fuel_consumption, ".4f")
            output_text.insert(tk.END, f"Unit: {unit}, Allocated Demand: {allocated_demand}, Fuel Rate: {fuel_rate}, Fuel Consumption: {fuel_consumption}\n")
        min_fuel_cost = format(min_fuel_cost, ".4f")
        output_text.insert(tk.END, f"Total Fuel Consumption: {min_fuel_cost}\n")

tk.Button(frame, text="Run Simulation", command=run_simulation).grid(row=unit_count + 1, column=0, columnspan=2)

output_text = tk.Text(root, width=100, height=20)
output_text.pack(pady=10)

root.mainloop()


In [1]:
import tkinter as tk
from tkinter import ttk
import itertools

unit_data = [
    "1,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "2,y = (7E-07*x**2) - (0.0005*x) + 0.3874,0,0",
    "3,y = (-5E-08*x**2) - (0.0001*x) + 0.315,150,350",
    "4,y = (7E-07*x**2) - (0.0005*x) + 0.3869,0,0",
    "5,y = (-5E-07*x**2) + (0.0003*x)+ 0.2885,90,300",
    "6,y = (-4E-06*x**2) + (0.0015*x) + 0.2112,150,330",
    "7,y = (1E-06*x**2) - (0.0006*x) + 0.3904,90,300",
    "8,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "9,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
    "10,y = (-5E-08*x**2) - (0.0001*x) + 0.315,0,0",
]

def fuel_rate_function(fuel_rate_str):
    def fuel_rate(x):
        return eval(fuel_rate_str)
    return fuel_rate

def calc_fuel_consumption(unit, demand):
    fuel_rate = operational_units[unit]["fuel_rate"](demand)
    fuel_consumption = fuel_rate * demand
    return fuel_rate, fuel_consumption

def find_optimal_dispatch(total_demand):
    optimal_dispatch = None
    min_fuel_cost = float("inf")
    closest_demand_diff = float("inf")

    for i in range(1, len(operational_units) + 1):
        for combination in itertools.combinations(operational_units.keys(), i):
            min_capacity = sum(operational_units[unit]["min_capacity"] for unit in combination)
            max_capacity = sum(operational_units[unit]["max_capacity"] for unit in combination)

            if min_capacity <= total_demand <= max_capacity:
                fuel_cost = 0
                demand_allocation = []
                remaining_demand = total_demand
                allocated_total_demand = 0

                for unit in combination:
                    if operational_units[unit]["max_capacity"] == 0:
                        continue
                    allocated_demand = max(
                        operational_units[unit]["min_capacity"],
                        min(operational_units[unit]["max_capacity"], remaining_demand)
                    )
                    remaining_demand -= allocated_demand
                    allocated_total_demand += allocated_demand
                    demand_allocation.append((unit, allocated_demand))

                    _, fuel_consumption = calc_fuel_consumption(unit, allocated_demand)
                    fuel_cost += fuel_consumption

                demand_diff = abs(allocated_total_demand - total_demand)
                if demand_diff < closest_demand_diff or (demand_diff == closest_demand_diff and fuel_cost < min_fuel_cost):
                    closest_demand_diff = demand_diff
                    min_fuel_cost = fuel_cost
                    optimal_dispatch = demand_allocation

    return optimal_dispatch, min_fuel_cost

units = {}
operational_units = {}

root = tk.Tk()
root.title("FRIP - Fuel Rate Improvement Program: Economic Load Dispatch Program")

# Adjust the window size
root.geometry("1400x1050")

# Add titles
title_font = ("Arial", 18, "bold")
subtitle_font = ("Arial", 14)

title = tk.Label(root, text="Fuel Rate Improvement Program (FRIP)", font=title_font, fg='dark green')
title.pack(pady=10)

subtitle = tk.Label(root, text="Economic Load Dispatch Program", font=subtitle_font, fg='red')
subtitle.pack(pady=5)

# Create Notebook (tabs)
notebook = ttk.Notebook(root)
notebook.pack(pady=10, expand=True)

# Create frames for each tab
frame_demand = ttk.Frame(notebook, width=400, height=400)
frame_units = ttk.Frame(notebook, width=400, height=400)

frame_demand.pack(fill='both', expand=True)
frame_units.pack(fill='both', expand=True)

notebook.add(frame_demand, text='24-Hour Demand Input')
notebook.add(frame_units, text='Unit Operational Status')

# Frame for 24-hour demand input
demand_labels = []
demand_entries = []
for hour in range(24):
    label = tk.Label(frame_demand, text=f"Hour {hour + 1} Demand:")
    label.grid(row=hour, column=0, padx=10, pady=5)
    entry = tk.Entry(frame_demand)
    entry.grid(row=hour, column=1, padx=10, pady=5)
    demand_labels.append(label)
    demand_entries.append(entry)

# Frame for unit operational status input
unit_labels = []
unit_entries = []
unit_count = len(unit_data)
for i in range(1, unit_count + 1):
    unit_number, _, _, _ = unit_data[i - 1].split(',')
    label = tk.Label(frame_units, text=f"Is unit {unit_number} operational? (1 for yes, 0 for no): ")
    label.grid(row=i - 1, column=0, padx=10, pady=5)
    entry = tk.Entry(frame_units)
    entry.grid(row=i - 1, column=1, padx=10, pady=5)
    unit_labels.append(label)
    unit_entries.append(entry)

def run_simulation():
    global units
    global operational_units
    units.clear()
    operational_units.clear()

    for i, data in enumerate(unit_data, start=1):
        try:
            unit_number, fuel_rate_str, min_capacity, max_capacity = data.split(',')
            unit_number = int(unit_number)
            min_capacity = float(min_capacity)
            max_capacity = float(max_capacity)
            operational = int(unit_entries[i - 1].get())
            units[unit_number] = {
                "fuel_rate": fuel_rate_function(fuel_rate_str[4:]),
                "min_capacity": min_capacity,
                "max_capacity": max_capacity,
                "operational": operational,
            }
        except ValueError:
            output_text.insert(tk.END, f"Invalid input for unit {unit_number}. Please enter 1 or 0 for operational status.\n")
            return

    operational_units = {k: v for k, v in units.items() if v["operational"]}

    try:
        total_demand = sum(float(entry.get()) for entry in demand_entries)
    except ValueError:
        output_text.insert(tk.END, "Invalid total demand input. Please enter numerical values for all hours.\n")
        return

    if not operational_units:
        output_text.insert(tk.END, "No operational units available.\n")
        return

    optimal_dispatch, min_fuel_cost = find_optimal_dispatch(total_demand)

    output_text.insert(tk.END, "Most economical unit(s) to operate:\n")
    if optimal_dispatch is None:
        output_text.insert(tk.END, "No feasible dispatch found for the given total demand.\n")
    else:
        for unit, allocated_demand in optimal_dispatch:
            fuel_rate, fuel_consumption = calc_fuel_consumption(unit, allocated_demand)
            fuel_rate = format(fuel_rate, ".4f")
            fuel_consumption = format(fuel_consumption, ".4f")
            output_text.insert(tk.END, f"Unit: {unit}, Allocated Demand: {allocated_demand}, Fuel Rate: {fuel_rate}, Fuel Consumption: {fuel_consumption}\n")
        min_fuel_cost = format(min_fuel_cost, ".4f")
        output_text.insert(tk.END, f"Total Fuel Consumption: {min_fuel_cost}\n")

# Add Run Simulation button
tk.Button(root, text="Run Simulation", command=run_simulation).pack(pady=10)

# Output text box
output_text = tk.Text(root, width=100, height=20)
output_text.pack(pady=10)

root.mainloop()
